In [8]:
import pandas as pd

# Paths
PROC_PATH = "../data/processed/v2/development_token_nohref_in_text.csv"
RAW_PATH  = "../data/raw/development.csv"
OUT_PATH  = "../data/processed/v3/development_v3.csv"

# Load
df_proc = pd.read_csv(PROC_PATH)
df_raw  = pd.read_csv(RAW_PATH)

# Keep only what serve dal raw
df_raw = df_raw[["Id", "article"]]

# Merge
df = df_proc.merge(df_raw, on="Id", how="inner")

# Drop old text column
if "text" in df.columns:
	df = df.drop(columns=["text"])

# Reorder columns (pulizia mentale)
cols = [
	"Id",
	"article",
	"title",
	"source",
	"n_tokens",
	"title_ratio",
	"year",
	"month",
	"has_timestamp",
	"log_n_links",
	"source_entropy",
	"source_max_prior",
	"source_support",
	"label"
]
df = df[cols]
df.columns


Index(['Id', 'article', 'title', 'source', 'n_tokens', 'title_ratio', 'year',
       'month', 'has_timestamp', 'log_n_links', 'source_entropy',
       'source_max_prior', 'source_support', 'label'],
      dtype='object')

In [9]:

# Save v3
df.to_csv(OUT_PATH, index=False)

print("[OK] Dataset v3 salvato:", OUT_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

[OK] Dataset v3 salvato: ../data/processed/v3/development_v3.csv
Shape: (79996, 14)
Columns: ['Id', 'article', 'title', 'source', 'n_tokens', 'title_ratio', 'year', 'month', 'has_timestamp', 'log_n_links', 'source_entropy', 'source_max_prior', 'source_support', 'label']


In [12]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Load v3
df = pd.read_csv("../data/processed/v3/development_v3.csv")
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
X = df.drop(columns=["label"])
y = df["label"]

text_col = "article"
num_cols = [
	"n_tokens",
	"title_ratio",
	"log_n_links",
	"source_entropy",
	"source_max_prior",
	"source_support",
	"year",
	"month",
	"has_timestamp"
]
cat_cols = ["source"]

preprocess = ColumnTransformer(
	transformers=[
		("text", TfidfVectorizer(
			max_features=50000,
			ngram_range=(1,2),
			min_df=3,
			max_df=0.9,
			stop_words="english"
		), text_col),
		("num", StandardScaler(), num_cols),
		("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
	],
	n_jobs=-1
)

model = Pipeline([
	("prep", preprocess),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y.iloc[tr])
	yp = model.predict(X.iloc[te])
	f1s.append(f1_score(y.iloc[te], yp, average="macro"))

print("BASELINE Macro F1:", np.mean(f1s))


BASELINE Macro F1: 0.6963069692007107


In [13]:
from sklearn.metrics import f1_score
from copy import deepcopy

def permutation_importance(model, X, y, feature, n_repeats=3):
	
	base_scores = []
	skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

	for tr, te in skf.split(X, y):
		model.fit(X.iloc[tr], y.iloc[tr])
		yp = model.predict(X.iloc[te])
		base_scores.append(f1_score(y.iloc[te], yp, average="macro"))

	base = np.mean(base_scores)
	drops = []

	for _ in range(n_repeats):
		X_perm = X.copy()
		X_perm[feature] = np.random.permutation(X_perm[feature].values)

		scores = []
		for tr, te in skf.split(X_perm, y):
			model.fit(X_perm.iloc[tr], y.iloc[tr])
			yp = model.predict(X_perm.iloc[te])
			scores.append(f1_score(y.iloc[te], yp, average="macro"))

		drops.append(base - np.mean(scores))

	return np.mean(drops)


for f in [
	"source_entropy",
	"source_max_prior",
	"source_support",
	"n_tokens",
	"title_ratio"
]:
	drop = permutation_importance(model, X, y, f)
	print(f"{f:20s} → ΔF1 = {drop:.4f}")


source_entropy       → ΔF1 = 0.0002
source_max_prior     → ΔF1 = 0.0002
source_support       → ΔF1 = 0.0004
n_tokens             → ΔF1 = 0.0006
title_ratio          → ΔF1 = 0.0018
